## Main type of tools:
### 1. Functional tools
    - convert any python tools into a tool where the agent can use
### 2. Query engine tools
    - let the agent use query engine
### 3. Toolspace
    - set of tools created by community
### 4. Utility Tools
    - special tools that manage large amount of data from other tools

In [18]:
# !pip install llama-index-llms-huggingface-api llama-index-embeddings-huggingface

In [19]:
# setup LM studio

from llama_index.llms.openai import OpenAI
llm = OpenAI(
    api_base = "http://localhost:1234/v1",
    api_key="lm-studio",
    model="local-model",
)

In [20]:
# setup function tool 
from llama_index.core.tools import FunctionTool

def get_weather(location: str) -> str:
    """Useful for getting the weather for a given location."""
    print(f"Getting weather for {location}")
    return f"The weather in {location} is sunny"

tool = FunctionTool.from_defaults(
    get_weather,
    name="my_weather_tool",
    description="Useful for getting the weather for a given location.",
)

tool.call("New York")

Getting weather for New York


ToolOutput(blocks=[TextBlock(block_type='text', text='The weather in New York is sunny')], tool_name='my_weather_tool', raw_input={'args': ('New York',), 'kwargs': {}}, raw_output='The weather in New York is sunny', is_error=False)

In [21]:
# setup query engine tool
from llama_index.core import VectorStoreIndex
from llama_index.core.tools import QueryEngineTool
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

embed_model = HuggingFaceEmbedding("BAAI/bge-small-en-v1.5")

db = chromadb.PersistentClient(path="./alfred_chroma_db") # create a vector database path = "./alfred_chroma_db"
chroma_collection = db.get_or_create_collection("alfred") # get or create a new database call alfred
vector_store = ChromaVectorStore(chroma_collection = chroma_collection) # connect chromadb and llamaindex

index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model) # use embedding model to create index
query_engine = index.as_query_engine(llm=llm) # llm now bonds with the index to become query engine

tool = QueryEngineTool.from_defaults(query_engine, name = "alfred db", description = "This database contain everything alfred needs to serve Batman") # wrap everyting into a QueryEngineTool with self define name and description



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

When you have a lot of tools, and you want to wrap them up for some specific agent, you might use toolspecs.

In [22]:
# pip install llama-index-tools-google

In [23]:
# setup toolspaces
from llama_index.tools.google import GmailToolSpec

tool_spec = GmailToolSpec() # gmail tools
tool_spec_list = tool_spec.to_tool_list() # convert the tool spec into a functionTool list so that the agent can read the name and description of the functions

In [24]:
[(tool.metadata.name, tool.metadata.description) for tool in tool_spec_list] # check the metadata in the tool_spec_list

[('load_data',
  "load_data() -> List[llama_index.core.schema.Document]\nLoad emails from the user's account."),
 ('search_messages',
  "search_messages(query: str, max_results: Optional[int] = None)\n\n        Searches email messages given a query string and the maximum number\n        of results requested by the user\n           Returns: List of relevant message objects up to the maximum number of results.\n\n        Args:\n            query (str): The user's query\n            max_results (Optional[int]): The maximum number of search results\n            to return."),
 ('create_draft',
  "create_draft(to: Optional[List[str]] = None, subject: Optional[str] = None, message: Optional[str] = None) -> str\n\n        Create and insert a draft email.\n           Print the returned draft's message and id.\n           Returns: Draft object, including draft id and message meta data.\n\n        Args:\n            to (Optional[str]): The email addresses to send the message to\n            subje

In [25]:
# !pip install llama-index-tools-mcp

In [26]:
# MCP setup

from llama_index.tools.mcp import BasicMCPClient, McpToolSpec

mcp_client = BasicMCPClient("http://127.0.0.1:8000/sse")
mcp_tool = McpToolSpec(client = mcp_client)

agent = await get_agent(mcp_tool)

agent_context = Context(agent)

ImportError: cannot import name 'StdioServerParameters' from partially initialized module 'mcp.client.stdio' (most likely due to a circular import) (c:\Python312\Lib\site-packages\mcp\client\stdio\__init__.py)

Didn't make a mcp server yet, maybe circle back later:
https://huggingface.co/learn/mcp-course/unit0/introduction

### Utility tools

1. `OnDemandToolLoader`: Automatically handles the embedding, builds a temporary RAG database in the background, search it, and it hands in the final answer back to the agent. Good for reading massive data for one question.
2. `LoadAndSearchToolSpace`: Seperate the load data and query part. Suitable for loading massive data once, then answer multiple questions.